In [1]:
%pip install -U ripser persim diffusers transformers accelerate sentencepiece protobuf

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os
import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
from tqdm import tqdm
from dataclasses import dataclass
from typing import Dict, List, Optional
from diffusers import PixArtSigmaPipeline, AutoencoderKL

# ============================================================
# 1. Configuration
# ============================================================

@dataclass
class ModelCfg:
    model_id: str = "PixArt-alpha/PixArt-Sigma-XL-2-1024-MS"
    short_name: str = "PixArt-Sigma"
    dtype: torch.dtype = torch.float16
    
    # Diffusion Specifics
    target_inference_step: int = 5    # Analyze at step 5 of 20
    total_inference_steps: int = 20
    resolution: int = 1024

@dataclass
class AnalysisConfig:
    n_samples: int = 20         # Total samples per category
    batch_size: int = 2         
    seed: int = 42
    
    # Toggle Analysis Modules
    compute_geometry: bool = True     # LID, Isotropy
    compute_component_balance: bool = True 
    compute_pixel_metrics: bool = True

def set_seed(seed: int):
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

# ============================================================
# 2. Robust Math & Metrics Library
# ============================================================

class MetricsLib:
    @staticmethod
    def entropy(probs, dim=-1, epsilon=1e-10):
        """Computes Shannon Entropy safely in float32."""
        probs = probs.float() 
        return -torch.sum(probs * torch.log(probs + epsilon), dim=dim)

    @staticmethod
    def gini_sparsity(tensor):
        """Gini Index: 0.0 (Equal) to 1.0 (Sparse)"""
        x = tensor.abs().float()
        n = x.shape[1]
        x_sorted, _ = torch.sort(x, dim=1)
        index = torch.arange(1, n + 1, device=x.device).float()
        return (n + 1 - 2 * torch.sum((n + 1 - index) * x_sorted, dim=1) / (torch.sum(x_sorted, dim=1) + 1e-9)) / n

    @staticmethod
    def compute_lid(data, k=20):
        """Robust Local Intrinsic Dimensionality (LID)."""
        data = data.double() # Force Double Precision
        N = data.shape[0]
        
        # Safety: Need at least k+1 samples (self + k neighbors)
        if N <= k: 
            # If we have too few samples, adjust k down
            if N > 2:
                k = N - 2
            else:
                return 0.0
        
        # Add tiny noise to prevent distance=0
        data = data + torch.randn_like(data) * 1e-6
        
        try:
            dist = torch.cdist(data, data, p=2)
            # k+1 because self is included
            topk_vals, _ = torch.topk(dist, k=k+1, largest=False)
            r = topk_vals[:, 1:] # Drop self
            
            r_max = r[:, -1].unsqueeze(1)
            r_max = torch.clamp(r_max, min=1e-9) 
            
            r_norm = r / r_max
            r_norm = torch.clamp(r_norm, min=1e-9)
            
            lids = -1 / (torch.mean(torch.log(r_norm), dim=1))
            val = lids.mean().item()
            return val if not np.isnan(val) else 0.0
        except Exception as e:
            # Fallback if calculation fails
            return 0.0

    @staticmethod
    def compute_explained_variance(data, threshold=0.9):
        """Effective dimensionality via PCA."""
        data = data.float()
        if data.shape[0] < 5: return 0
        
        centered = data - data.mean(dim=0)
        try:
            _, S, _ = torch.svd(centered)
            total_var = S.sum() + 1e-9
            cumsum = torch.cumsum(S, dim=0) / total_var
            # Find index where cumsum > threshold
            dim_90 = (cumsum > threshold).nonzero()
            if len(dim_90) > 0:
                return dim_90[0].item()
            return data.shape[1]
        except:
            return 0

# ============================================================
# 3. Data & Model
# ============================================================

def get_datasets(cfg: AnalysisConfig) -> Dict[str, List[str]]:
    
    datasets = {}
    print(f"Loading Image Prompts (N={cfg.n_samples})...")
    
    # --- 1. Coherent ---
    subjects = ["cat", "car", "forest", "city", "apple", "robot", "mountain", "ocean", "tiger", "library"]
    # Use standard random choice without fixed seed
    datasets['1_Coherent'] = [
        f"A photo of a {np.random.choice(subjects)}, 4k, highly detailed." 
        for _ in range(cfg.n_samples)
    ]

    # --- 2. Paradoxical ---
    csv_path = "paradox.csv"
    
    # Force check absolute path if needed, or print current directory
    if not os.path.exists(csv_path):
        print(f"⚠️  WARNING: '{csv_path}' not found in {os.getcwd()}")
        print("   -> Using fallback list (only 3 prompts). Check your file path!")
        fallback_list = ["square circle", "frozen fire", "transparent rock"]
        datasets['2_Paradoxical'] = [f"A surreal photo of {np.random.choice(fallback_list)}" for _ in range(cfg.n_samples)]
        
    else:
        try:
            # Load CSV
            df = pd.read_csv(csv_path, header=None)
            all_prompts = df.iloc[:, 0].astype(str).tolist()
            
            # Clean empty lines if any
            all_prompts = [p for p in all_prompts if len(p) > 3]
            
            # Shuffle the entire list randomly
            np.random.shuffle(all_prompts)
            
            # Take the first N samples
            if len(all_prompts) >= cfg.n_samples:
                selected_prompts = all_prompts[:cfg.n_samples]
            else:
                # If we don't have enough, cycle them
                print(f"   Note: CSV has {len(all_prompts)} prompts, repeating to fill {cfg.n_samples}.")
                selected_prompts = (all_prompts * (cfg.n_samples // len(all_prompts) + 1))[:cfg.n_samples]
            
            datasets['2_Paradoxical'] = selected_prompts
            print(f"  -> ✅ Successfully loaded {len(selected_prompts)} prompts from {csv_path}")
            
        except Exception as e:
            print(f"❌ Error reading CSV: {e}")
            datasets['2_Paradoxical'] = [f"Error Prompt {i}" for i in range(cfg.n_samples)]

    return datasets

def find_layers(pipeline):
    model = pipeline.transformer
    return model.transformer_blocks, "attn1", "attn2", "ff"

# ============================================================
# 4. Main Analysis Engine
# ============================================================

def analyze_dynamics(model_cfg: ModelCfg, exp_cfg: AnalysisConfig, datasets: Dict):
    print(f"\n{'='*60}")
    print(f"🧠 PIXART SIGMA DEEP SCAN")
    print(f"{'='*60}")

    # Load Model
    pipe = PixArtSigmaPipeline.from_pretrained(model_cfg.model_id, torch_dtype=model_cfg.dtype, use_safetensors=True).to("cuda")
    pipe.set_progress_bar_config(disable=True)
    
    layers, attn1_name, attn2_name, ff_name = find_layers(pipe)
    n_layers = len(layers)
    
    # --- Helper: Run Diffusion Step ---
    def run_to_step(batch_prompts):
        with torch.no_grad():
            prompt_embeds, prompt_attn_mask, _, _ = pipe.encode_prompt(batch_prompts, do_classifier_free_guidance=False)
            bsz = len(batch_prompts)
            latents = torch.randn((bsz, pipe.transformer.config.in_channels, model_cfg.resolution//8, model_cfg.resolution//8), device=pipe.device, dtype=pipe.dtype)
            
            pipe.scheduler.set_timesteps(model_cfg.total_inference_steps, device=pipe.device)
            cond_kwargs = {
                "resolution": torch.tensor([model_cfg.resolution]*2, device=pipe.device, dtype=pipe.dtype).repeat(bsz, 1),
                "aspect_ratio": torch.tensor([1.], device=pipe.device, dtype=pipe.dtype).repeat(bsz, 1)
            }

            for i, t in enumerate(pipe.scheduler.timesteps):
                ts_batch = t.unsqueeze(0).expand(bsz)
                if i == model_cfg.target_inference_step:
                    return latents, ts_batch, prompt_embeds, prompt_attn_mask, cond_kwargs
                
                noise_pred = pipe.transformer(latents, timestep=ts_batch, encoder_hidden_states=prompt_embeds, encoder_attention_mask=prompt_attn_mask, added_cond_kwargs=cond_kwargs, return_dict=False)[0]
                if noise_pred.shape[1] == 2*latents.shape[1]: noise_pred, _ = noise_pred.chunk(2, dim=1)
                latents = pipe.scheduler.step(noise_pred, t, latents).prev_sample
        return None

    # --- Pass 1: Boundary Vectors ---
    print("\n🚀 Pass 1: Calculating Boundary Vectors...")
    mean_storage = {l: {'1_Coherent': None, '2_Paradoxical': None} for l in range(n_layers)}
    counts = {l: {'1_Coherent': 0, '2_Paradoxical': 0} for l in range(n_layers)}

    for bucket, prompts in datasets.items():
        for i in range(0, len(prompts), exp_cfg.batch_size):
            batch_txt = prompts[i:i+exp_cfg.batch_size]
            run_data = run_to_step(batch_txt)
            if not run_data: continue
            latents, t, embeds, mask, cond = run_data
            
            # Simple Hook for Mean
            layer_means = {}
            def get_mean_hook(l):
                return lambda m, i, o: layer_means.update({l: o.mean(dim=1).detach()}) 
            
            hooks = [layers[l].register_forward_hook(get_mean_hook(l)) for l in range(n_layers)]
            with torch.no_grad(): _ = pipe.transformer(latents, encoder_hidden_states=embeds, encoder_attention_mask=mask, timestep=t, added_cond_kwargs=cond)
            for h in hooks: h.remove()

            for l, val in layer_means.items():
                if mean_storage[l][bucket] is None: mean_storage[l][bucket] = val.sum(dim=0).float()
                else: mean_storage[l][bucket] += val.sum(dim=0).float()
                counts[l][bucket] += val.shape[0]

    boundary_vectors = {}
    for l in range(n_layers):
        if counts[l]['1_Coherent'] > 0 and counts[l]['2_Paradoxical'] > 0:
            mu_f = mean_storage[l]['1_Coherent'] / counts[l]['1_Coherent']
            mu_p = mean_storage[l]['2_Paradoxical'] / counts[l]['2_Paradoxical']
            vec = mu_p - mu_f
            boundary_vectors[l] = (vec / (vec.norm() + 1e-9)).to(pipe.device).to(pipe.dtype)
    del mean_storage, counts

    # --- Pass 2: Deep Metrics Scan ---
    print("\n🚀 Pass 2: Deep Metrics Scan...")
    results = []
    geo_storage = {l: {'1_Coherent': [], '2_Paradoxical': []} for l in range(n_layers)}

    for bucket, prompts in datasets.items():
        for i in tqdm(range(0, len(prompts), exp_cfg.batch_size), desc=f"  {bucket}"):
            batch_txt = prompts[i:i+exp_cfg.batch_size]
            run_data = run_to_step(batch_txt)
            if not run_data: continue
            latents, t, embeds, mask, cond = run_data
            
            batch_data = {l: {} for l in range(n_layers)}
            
            # Capture block output
            def get_block_hook(l):
                def hook(module, args, output):
                    batch_data[l]['output'] = output.detach() 
                return hook
            
            # Capture component magnitudes
            def get_sub_hook(l, name):
                def hook(module, args, output):
                    if isinstance(output, tuple): output = output[0]
                    batch_data[l][name] = output.norm(dim=-1).mean(dim=1).detach()
                return hook

            hooks = []
            for l in range(n_layers):
                hooks.append(layers[l].register_forward_hook(get_block_hook(l)))
                if exp_cfg.compute_component_balance:
                    hooks.append(getattr(layers[l], attn1_name).register_forward_hook(get_sub_hook(l, 'attn_self')))
                    hooks.append(getattr(layers[l], attn2_name).register_forward_hook(get_sub_hook(l, 'attn_cross')))
                    hooks.append(getattr(layers[l], ff_name).register_forward_hook(get_sub_hook(l, 'mlp')))

            with torch.no_grad():
                _ = pipe.transformer(latents, encoder_hidden_states=embeds, encoder_attention_mask=mask, timestep=t, added_cond_kwargs=cond)
            for h in hooks: h.remove()
            
            bsz = latents.shape[0]
            for l in range(n_layers):
                if 'output' not in batch_data[l]: continue
                
                # Spatial Mean for Geometry
                act_spatial = batch_data[l]['output'] 
                act_mean = act_spatial.mean(dim=1)    
                
                # Store for LID
                geo_storage[l][bucket].append(act_mean.float().cpu())
                
                for b_idx in range(bsz):
                    row = {"bucket": bucket, "layer": l, "relative_depth": l/n_layers}
                    vec_f32 = act_mean[b_idx].float()
                    
                    # 1. Entropy & Sparsity
                    probs = F.softmax(vec_f32, dim=-1)
                    row['feature_entropy'] = MetricsLib.entropy(probs).item()
                    row['gini_sparsity'] = MetricsLib.gini_sparsity(vec_f32.unsqueeze(0)).mean().item()
                    
                    # 2. Alignment
                    if l in boundary_vectors:
                        b_vec = boundary_vectors[l].float()
                        row['confusion_alignment'] = (vec_f32 @ b_vec).item()
                        row['signal_magnitude'] = vec_f32.norm().item()
                    else:
                        row['confusion_alignment'] = 0.0
                        row['signal_magnitude'] = 0.0

                    # 3. Component Balance
                    if exp_cfg.compute_component_balance:
                        s_mag = batch_data[l]['attn_self'][b_idx].float()
                        c_mag = batch_data[l]['attn_cross'][b_idx].float()
                        m_mag = batch_data[l]['mlp'][b_idx].float()
                        total = s_mag + c_mag + m_mag + 1e-9
                        
                        row['ratio_self_attn'] = (s_mag / total).item()
                        row['ratio_cross_attn'] = (c_mag / total).item()
                        row['ratio_mlp'] = (m_mag / total).item()

                    # 4. Pixel Proxy (Spatial Variance)
                    if exp_cfg.compute_pixel_metrics:
                        spatial_var = act_spatial[b_idx].var(dim=0).mean().item()
                        row['visual_complexity_proxy'] = spatial_var
                        
                    results.append(row)

    # --- Pass 3: Geometry ---
    print("\n📐 Calculating Global Geometry (LID, PCA)...")
    geo_results = []
    
    for l in tqdm(range(n_layers)):
        for bucket in ['1_Coherent', '2_Paradoxical']:
            vecs = geo_storage[l][bucket]
            if len(vecs) == 0: continue
            
            # Use CAT instead of STACK to create a single 2D tensor (Total_Samples, Dim)
            data = torch.cat(vecs, dim=0).float() 
            
            g_row = {"layer": l, "bucket": bucket}
            # Calculate K dynamically based on total samples available
            safe_k = min(10, data.shape[0] - 2) 
            if safe_k < 2: safe_k = 2
            
            g_row['LID'] = MetricsLib.compute_lid(data, k=safe_k)
            g_row['explained_variance_dim'] = MetricsLib.compute_explained_variance(data)
            
            geo_results.append(g_row)

    # --- Merge ---
    df_main = pd.DataFrame(results)
    df_geo = pd.DataFrame(geo_results)
    
    meta_keys = ['bucket', 'layer', 'relative_depth']
    numeric_cols = [c for c in df_main.columns if c not in meta_keys]
    df_agg = df_main.groupby(meta_keys)[numeric_cols].mean().reset_index()
    
    final_df = pd.merge(df_agg, df_geo, on=['layer', 'bucket'], how='left')
    final_df = final_df.fillna(0.0)
    final_df.to_csv("pixart_deep_analysis.csv", index=False)
    
    return final_df

# ============================================================
# 5. Visual Generator
# ============================================================

def generate_visual_samples(model_cfg: ModelCfg, datasets: Dict, n_show: int = 4):
    print(f"\n🎨 GENERATING SAMPLES (N={n_show})...")
    output_dir = "visual_samples"
    os.makedirs(output_dir, exist_ok=True)
    
    pipe = PixArtSigmaPipeline.from_pretrained(
        model_cfg.model_id, torch_dtype=model_cfg.dtype, use_safetensors=True
    ).to("cuda")
    pipe.set_progress_bar_config(disable=True)
    
    for bucket, prompts in datasets.items():
        print(f"  Generating {bucket}...")
        subset = prompts[:n_show]
        for i, prompt in enumerate(subset):
            image = pipe(prompt, num_inference_steps=20).images[0]
            clean_name = bucket.replace(" ", "_")
            fname = f"{output_dir}/{clean_name}_{i}.png"
            image.save(fname)
            print(f"    -> Saved {fname}")
    print("✅ Visuals Done.")

if __name__ == "__main__":
    import time
    random_seed = int(time.time())

    cfg = AnalysisConfig(n_samples=200, batch_size=2, seed=random_seed) 
    
    set_seed(cfg.seed) 
    
    model_cfg = ModelCfg()
    
    datasets = get_datasets(cfg)
    
    try:
        # 1. Run Analysis
        df = analyze_dynamics(model_cfg, cfg, datasets)
        
        # 2. Print Sample
        print("\nSample Data (Layer 14):")
        cols = ['bucket', 'feature_entropy', 'LID', 'ratio_cross_attn', 'visual_complexity_proxy']
        print(df[df['layer'] == 14][cols])
        
        # 3. Generate Pictures
        generate_visual_samples(model_cfg, datasets)
        
    except Exception as e:
        print(f"Error: {e}")
        import traceback
        traceback.print_exc()

Loading Image Prompts (N=50)...
  -> ✅ Successfully loaded 50 prompts from paradox.csv

🧠 PIXART SIGMA DEEP SCAN


Loading pipeline components...: 100%|██████████| 5/5 [00:01<00:00,  3.11it/s]



🚀 Pass 1: Calculating Boundary Vectors...

🚀 Pass 2: Deep Metrics Scan...


  2_Paradoxical: 100%|██████████| 25/25 [00:20<00:00,  1.20it/s]



📐 Calculating Global Geometry (LID, PCA)...


...

    -> Saved visual_samples/2_Paradoxical_0.png



Setting `clean_caption=True` requires the ftfy library but it was not found in your environment. Checkout the instructions on the
installation section: https://github.com/rspeer/python-ftfy/tree/master#installing and follow the ones
that match your environment. Please note that you may need to restart your runtime after installation.

Setting `clean_caption` to False...

Setting `clean_caption=True` requires the ftfy library but it was not found in your environment. Checkout the instructions on the
installation section: https://github.com/rspeer/python-ftfy/tree/master#installing and follow the ones
that match your environment. Please note that you may need to restart your runtime after installation.

Setting `clean_caption` to False...


    -> Saved visual_samples/2_Paradoxical_1.png



Setting `clean_caption=True` requires the ftfy library but it was not found in your environment. Checkout the instructions on the
installation section: https://github.com/rspeer/python-ftfy/tree/master#installing and follow the ones
that match your environment. Please note that you may need to restart your runtime after installation.

Setting `clean_caption` to False...

Setting `clean_caption=True` requires the ftfy library but it was not found in your environment. Checkout the instructions on the
installation section: https://github.com/rspeer/python-ftfy/tree/master#installing and follow the ones
that match your environment. Please note that you may need to restart your runtime after installation.

Setting `clean_caption` to False...


    -> Saved visual_samples/2_Paradoxical_2.png
    -> Saved visual_samples/2_Paradoxical_3.png
✅ Visuals Done.
